# Agentic AI Design Patterns

## Overview

This notebook covers essential design patterns for building Agentic AI systems.

## 1. Reactive vs Planning Agents

### Reactive Agents

Reactive agents respond directly to stimuli without maintaining internal state or planning ahead.

**Characteristics:**
- Fast response times
- Simple implementation
- No long-term planning
- Limited to well-defined scenarios

### Planning Agents

Planning agents maintain internal models, consider future states, and develop step-by-step plans.

**Characteristics:**
- Can handle complex, multi-step tasks
- Maintains goal state
- Can replan when conditions change
- More sophisticated decision making

### When to Use Each

| Scenario | Agent Type |
|----------|------------|
| Simple FAQ | Reactive |
| Customer support triage | Reactive |
| Complex research task | Planning |
| Multi-step workflows | Planning |
| Real-time monitoring | Reactive |

In [ ]:
# Reactive Agent Implementation

from enum import Enum

class QueryType(Enum):
    ORDER_STATUS = "order_status"
    STORE_HOURS = "store_hours"
    RETURN_POLICY = "return_policy"
    COMPLEX = "complex"

# Mock data
ORDERS = {
    "ORD001": {"status": "Shipped", "items": ["Laptop"], "total": 1299.99},
    "ORD002": {"status": "Processing", "items": ["Mouse", "Keyboard"], "total": 159.98},
    "ORD003": {"status": "Delivered", "items": ["Monitor"], "total": 349.99},
}

STORE_INFO = {
    "hours": "Mon-Sat: 9AM-9PM, Sun: 10AM-6PM",
    "address": "123 Main St, City",
    "phone": "555-1234",
}


class ReactiveOrderAgent:
    """Reactive agent that handles simple order queries."""

    def classify_query(self, query):
        query_lower = query.lower()
        if "order status" in query_lower or "track" in query_lower:
            return QueryType.ORDER_STATUS
        elif "hours" in query_lower or "open" in query_lower:
            return QueryType.STORE_HOURS
        elif "return" in query_lower or "refund" in query_lower:
            return QueryType.RETURN_POLICY
        else:
            return QueryType.COMPLEX

    def handle_order_status(self, query):
        for order_id in ORDERS.keys():
            if order_id in query.upper():
                order = ORDERS[order_id]
                return f"Order {order_id}: {order['status']} - Items: {', '.join(order['items'])}"
        return "Please provide your order ID (e.g., ORD001)"

    def handle_store_hours(self, query):
        return STORE_INFO["hours"]

    def handle_return_policy(self, query):
        return "30-day return policy for most items. Receipt required."

    def handle_complex(self, query):
        return "This requires more detailed assistance."

    def process(self, query):
        query_type = self.classify_query(query)
        handlers = {
            QueryType.ORDER_STATUS: self.handle_order_status,
            QueryType.STORE_HOURS: self.handle_store_hours,
            QueryType.RETURN_POLICY: self.handle_return_policy,
            QueryType.COMPLEX: self.handle_complex,
        }
        return handlers[query_type](query)


# Test Reactive Agent
reactive = ReactiveOrderAgent()
queries = [
    "What's my order status for ORD001?",
    "What are your store hours?",
]

for q in queries:
    print(f"Query: {q}")
    print(f"Response: {reactive.process(q)}\n")

In [ ]:
# Planning Agent Implementation

from dataclasses import dataclass
from typing import List, Any

@dataclass
class PlanStep:
    step_id: int
    description: str
    status: str = "pending"
    result: Any = None


class PlanningTravelAgent:
    """Planning agent for complex travel booking."""

    def __init__(self):
        self.plan = []

    def create_plan(self, goal):
        self.plan = []
        if "book flight" in goal.lower():
            self.plan = [
                PlanStep(1, "Search available flights"),
                PlanStep(2, "Compare prices and times"),
                PlanStep(3, "Select best option"),
                PlanStep(4, "Collect passenger details"),
                PlanStep(5, "Process payment"),
                PlanStep(6, "Send confirmation"),
            ]
        return self.plan

    def execute_step(self, step_id):
        for step in self.plan:
            if step.step_id == step_id:
                step.status = "completed"
                step.result = f"Step {step_id} completed"
                return f"Executed: {step.description}"
        return f"Step {step_id} not found"

    def get_plan_status(self):
        status = "Plan Status:\n"
        for step in self.plan:
            status += f"{step.step_id}. {step.description} - {step.status}\n"
        return status


# Test Planning Agent
planner = PlanningTravelAgent()
planner.create_plan("Book a flight to NYC")
print(planner.get_plan_status())
planner.execute_step(1)
planner.execute_step(2)
print(planner.get_plan_status())

---

## 2. Reflection Patterns

Reflection allows agents to evaluate their own outputs and improve over time.

### Types of Reflection

1. **Self-Correction**: Identify and fix errors in reasoning
2. **Quality Assessment**: Evaluate output quality
3. **Strategy Adjustment**: Modify approach based on results

In [ ]:
# Reflection Implementation

class ReflectiveAgent:
    """Agent with reflection capabilities."""

    def __init__(self):
        self.quality_threshold = 0.7

    def reflect(self, response, context):
        # Check response length
        if len(response) < 10:
            return self.improve(response, "Too short")

        # Check for specific requirements
        if context.get("requires_order_id") and "ORD" not in response:
            return self.improve(response, "Missing required information")

        # Check quality
        quality = self.assess_quality(response)
        if quality < self.quality_threshold:
            return self.improve(response, "Low quality")

        return response

    def assess_quality(self, response):
        """Assess response quality (0-1)."""
        score = 0.5
        if len(response) > 20:
            score += 0.2
        if "." in response:
            score += 0.15
        if len(response) > 0 and response[0].isupper():
            score += 0.15
        return min(score, 1.0)

    def improve(self, response, reason):
        if "Too short" in reason:
            response += " Please let me know if you need more information."
        elif "Missing" in reason:
            response += " Could you please provide your order ID?"
        return response


# Test Reflection
reflector = ReflectiveAgent()
responses = [
    ("Hi", {"requires_order_id": False}),
    ("ORD001", {"requires_order_id": True}),
]

for resp, ctx in responses:
    print(f"Original: {resp}")
    print(f"Reflected: {reflector.reflect(resp, ctx)}\n")

---

## 3. Memory Patterns

### Types of Memory

- **Buffer Memory**: Stores most recent interactions verbatim
- **Sliding Window Memory**: Maintains fixed-size window of recent context
- **Summary Memory**: Compresses older interactions into summaries
- **Vector Memory**: Uses embeddings to retrieve relevant past context
- **Scratchpad**: Temporary working memory for intermediate calculations
- **Shared Memory**: Allows multiple agents to share context

In [ ]:
# Memory Implementations

class BufferMemory:
    """Buffer memory - stores recent interactions."""

    def __init__(self, capacity=10):
        self.capacity = capacity
        self.buffer = []

    def add(self, role, content):
        self.buffer.append({"role": role, "content": content})
        if len(self.buffer) > self.capacity:
            self.buffer.pop(0)

    def get_context(self):
        return "\n".join([f"{m['role']}: {m['content']}" for m in self.buffer])


class VectorMemory:
    """Vector-like memory for semantic retrieval."""

    def __init__(self):
        self.memories = []

    def add(self, key, content, importance=1):
        self.memories.append({"key": key, "content": content, "importance": importance})

    def retrieve(self, query, top_k=3):
        sorted_memories = sorted(
            self.memories, key=lambda x: x["importance"], reverse=True
        )
        return [m["content"] for m in sorted_memories[:top_k]]


# Test Memory
buffer = BufferMemory(capacity=3)
buffer.add("user", "I want to buy a laptop")
buffer.add("assistant", "What is your budget?")
buffer.add("user", "Around $1000")
buffer.add("user", "I also need a mouse")
print("Buffer Context:")
print(buffer.get_context())

vec_mem = VectorMemory()
vec_mem.add("preference", "Prefers Apple products", importance=3)
vec_mem.add("budget", "Budget is $1000", importance=5)
vec_mem.add("use_case", "Mainly for work", importance=1)
print("\nVector Memory Retrieval:")
for m in vec_mem.retrieve("laptop"):
    print(f"  - {m}")

---

## 4. Tools and Tool Integration

Tools extend agent capabilities beyond text generation.

### Tool Design Principles

1. **Single Responsibility**: Each tool does one thing well
2. **Clear Inputs/Outputs**: Well-defined interfaces
3. **Error Handling**: Graceful failure modes

### Tool Types

- **Information Retrieval**: Search, Database queries
- **Computation**: Calculator, Code execution
- **External APIs**: Weather, Stocks, News
- **File Operations**: Read, Write, List

In [ ]:
# Tool Example

def calculate(expression):
    """Evaluate a math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"


def search(query):
    """Search for information."""
    return f"Search results for: {query}"


# Example tool calls
print("calculate(5 + 3):", calculate("5 + 3"))
print("search('weather'):", search("weather"))

---

## 5. ReAct Pattern (Reasoning + Acting)

ReAct combines reasoning traces with actions.

### ReAct Loop

1. **Thought**: Reason about the current situation
2. **Action**: Execute an action
3. **Observation**: Observe the result
4. **Repeat**: Continue until goal is achieved

In [ ]:
# ReAct Implementation

class ReActAgent:
    """Agent implementing Reasoning + Acting pattern."""

    def __init__(self):
        self.trace = []

    def think(self, observation):
        """Generate thought based on observation."""
        self.trace.append(f"Observation: {observation}")
        return f"Based on: {observation}"

    def act(self, thought):
        """Execute action based on thought."""
        action = f"Action: {thought}"
        self.trace.append(action)
        return action

    def react(self, max_iterations=5):
        """Execute ReAct loop."""
        results = []
        for i in range(max_iterations):
            obs = f"Step {i + 1}"
            thought = self.think(obs)
            action = self.act(thought)
            results.append(f"{thought} -> {action}")
        return results


# Test ReAct
react_agent = ReActAgent()
for step in react_agent.react(3):
    print(step)

---

## 6. Combined Agent System

In [ ]:
# Combined Agent System

class HybridAgentSystem:
    """Combines reactive, planning, reflection, and memory."""

    def __init__(self):
        self.buffer_memory = []
        self.scratchpad = {}
        self.reflector = ReflectiveAgent()
        self.react_agent = ReActAgent()
        self.reactive_agent = ReactiveOrderAgent()
        self.planner = PlanningTravelAgent()

    def process(self, query, use_planning=False):
        # Add to memory
        self.buffer_memory.append({"role": "user", "content": query})

        if use_planning:
            response = self._plan(query)
        else:
            response = self._react(query)

        # Apply reflection
        response = self.reflector.reflect(response, {})

        # Store in memory
        self.buffer_memory.append({"role": "assistant", "content": response})

        return response

    def _react(self, query):
        """Use reactive agent."""
        return self.reactive_agent.process(query)

    def _plan(self, query):
        """Use planning agent with ReAct."""
        if "book" in query.lower():
            self.planner.create_plan(query)
            return f"Plan created with {len(self.planner.plan)} steps"
        return "Planning not needed for this query"


# Test Hybrid System
system = HybridAgentSystem()

print("=== Reactive Mode ===")
result = system.process("What's my order status?")
print(f"Result: {result}")

print("\n=== Planning Mode ===")
result = system.process("Book flight to NYC", use_planning=True)
print(f"Result: {result}")

print("\n=== Memory Contents ===")
for msg in system.buffer_memory:
    print(f"{msg['role']}: {msg['content']}")

---

## Quiz

### Question 1
What is the main characteristic of a reactive agent?

A) Maintains internal state and plans ahead
B) Responds directly to stimuli without planning
C) Uses complex reasoning
D) Can learn from past interactions

**Answer: B**

### Question 2
Which memory pattern stores only the most recent interactions?

A) Summary Memory
B) Vector Memory
C) Buffer Memory
D) Scratchpad

**Answer: C**

### Question 3
What does the "R" in ReAct stand for?

A) Retrieval
B) Reasoning
C) Response
D) Reflection

**Answer: B**

### Question 4
Which reflection type involves identifying and fixing errors?

A) Quality Assessment
B) Self-Correction
C) Strategy Adjustment
D) None of the above

**Answer: B**

### Question 5
When should you use a planning agent over a reactive agent?

A) For simple FAQ queries
B) For multi-step complex workflows
C) When response time is critical
D) For stateless interactions

**Answer: B**

### Question 6
What is the purpose of a scratchpad in agent memory?

A) Long-term storage of user preferences
B) Temporary working memory for calculations
C) Storing conversation history
D) Vector-based semantic retrieval

**Answer: B**

### Question 7
Which tool type would a weather API be classified as?

A) Information Retrieval
B) Computation
C) External API
D) File Operations

**Answer: C**

### Question 8
What is the key difference between sliding window and buffer memory?

A) Sliding window uses summaries
B) Buffer stores everything verbatim
C) They are essentially the same
D) Sliding window has fixed size, buffer can grow

**Answer: D**

### Question 9
In ReAct, what follows "Thought"?

A) Observation
B) Action
C) Reasoning
D) Planning

**Answer: B**

### Question 10
What is the primary benefit of using tools with agents?

A) Faster responses
B) Extended capabilities beyond text generation
C) Lower costs
D) Simpler implementation

**Answer: B**

---

## References

- ReAct Paper: https://arxiv.org/abs/2210.03629
- LangChain Agents: https://python.langchain.com/docs/modules/agents/
- Agent Memory Patterns: https://python.langchain.com/docs/modules/memory/
- Tool Use in Agents: https://python.langchain.com/docs/modules/agents/tools/
- Reflection Patterns: https://arxiv.org/abs/2210.05101